In [ ]:
# Cell 1 - Imports
from llama_cpp import Llama
from tqdm import tqdm
import pandas as pd
import numpy as np
import os
import torch
import ast

In [ ]:
# Cell 2 - Konfiguration
output_folder = "llm_templates/"
min_characters = 50

# Dehumanisierungs-Kategorien
setup_explicit = {
    "ex_animalistic": ["ein Tier", "eine unmenschliche Kreatur (nicht Tier)", "eine Krankheit (nicht Tier)"],
    "ex_mechanistic": ["ein unbelebter Gegenstand (nicht Tier)", "eine Maschine (nicht Tier)", 
                      "ein Roboter (nicht Tier)", "ein Werkzeug"],
}

setup_implicit = {
    "imp_animalistic": ["irrational", "unintelligent", "unkultiviert", "grob", "unreif", "ohne Moral"],
    "imp_mechanistic": ["träge", "kalt", "starr", "passiv", "oberflächlich", "ohne Handlungsfähigkeit"]
}

In [ ]:
# Cell 3 - Hilfsfunktionen
def ensure_folder_exists(folder_path):
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)

## load LLama3.1 model. We use a lobotomized model to avoid guardrails
def load_llm():
    llm = Llama(
        model_path = '../models/LLama3.1-lobo.gguf',
        n_gpu_layers = -1, 
        temperature = 0.7,
        n_ctx = 512, 
        logits_all = False,
        verbose = False)
    return llm

In [ ]:
# Cell 4 - Prompt-Funktion
def call_prompt(trait, dehum_type = "exp"):
    if dehum_type == "exp":
        query = (
            f'Wie vergleicht ein Social-Media-Nutzer eine Gruppe von Menschen oder einen anderen Nutzer auf negative Weise mit {trait}? '
            f'Verwenden Sie dabei gängige Twitter/Reddit-Sprache und Formulierungen. '
            f'Die Zielgruppen sollten häufig entmenschlichte Gruppen sein (verwenden Sie keine Platzhalter für Gruppen). '
            f'Bitte erstellen Sie 5 konkrete Beispiele und geben Sie sie als Liste aus. '
            f'Geben Sie ausschließlich die Liste der Beispiele aus und sonst nichts, in folgendem Format: ["Beispiel 1", "Beispiel 2", ...]. \n'
            f'Ausgabe:'
        )
    else:
        query = (
            f'Wie beschreibt ein Social-Media-Nutzer eine Gruppe von Menschen oder einen anderen Nutzer auf negative Weise als {trait}? '
            f'Die Zielgruppen sollten häufig entmenschlichte Gruppen sein (verwenden Sie keine Platzhalter für Gruppen). '
            f'Verwenden Sie dabei gängige Twitter/Reddit-Sprache und Formulierungen. '
            f'Bitte erstellen Sie 5 konkrete Beispiele und geben Sie sie als Liste aus. '
            f'Geben Sie ausschließlich die Liste der Beispiele aus und sonst nichts, in folgendem Format: ["Beispiel 1", "Beispiel 2", ...]. \n'
            f'Ausgabe:'
        )
    
    output = llm.create_chat_completion(
        messages=[
            {
                "role": "system", 
                "content": "Du bist eine sehr hilfsbereite Doktorandin. Deine Aufgabe besteht darin, Beispiele zu erstellen, die dabei helfen, Machine-Learning-Modelle zur Erkennung verschiedener Formen von Entmenschlichung (Dehumanisierung) zu trainieren."
            },
            {
                "role": "user",
                "content": query
            }
        ],
        max_tokens=-1
    )
    return output

In [ ]:
# Cell 5 - Kandidatengenerierung
def get_candidates(traits, reps=4, dehum_type="exp"):
    res_text = []
    res_trait = []
    for trait in traits:
        result_list = []
        counter = 0
        while counter < reps:
            output = call_prompt(trait, dehum_type=dehum_type)
            if output["usage"]["completion_tokens"] > min_characters:
                try:
                    erg = ast.literal_eval(output["choices"][0]["message"]["content"].replace("\n", ""))
                    result_list.extend(erg)
                    counter = counter + 1
                    print(f"Success {counter} for trait '{trait}'")
                    print(erg)
                except:
                    print("error")
            else:
                continue
        res_text.extend(result_list)
        res_trait.extend([trait] * len(result_list))
    return pd.DataFrame({"text": res_text, "dim": res_trait})


In [ ]:

# GPU Cache leeren
torch.cuda.empty_cache()

# LLM laden
llm = load_llm()

# Ausgabeordner erstellen
ensure_folder_exists(output_folder)

# Explizite Templates generieren
explicit_templates = pd.DataFrame()
for key in setup_explicit.keys():
    explicit_templates = pd.concat([explicit_templates, get_candidates(setup_explicit[key], 60, dehum_type="exp")])
explicit_templates.to_csv(os.path.join(output_folder, "Llama3.1_explicit_artificial_german.csv"), index=False)

# Implizite Templates generieren
implicit_templates = pd.DataFrame()
for key in setup_implicit.keys():
    implicit_templates = pd.concat([implicit_templates, get_candidates(setup_implicit[key], 20, dehum_type="imp")])
implicit_templates.to_csv(os.path.join(output_folder, "Llama3.1_implicit_artificial_german.csv"), index=False)
